# Train U-Net on miniature dataset
This notebook shows a minimal training loop using the `UNet2Dto3D` model and
HDF5 dataset generated by the simple simulator. It also includes utilities to
inspect data shapes, run a short training loop, and visualise example
reconstructions.

## Cell 1 — Set project root and verify dataset file

In [ ]:
import os
from pathlib import Path

# Walk up from CWD until we find pyproject.toml (= project root)
candidate = Path.cwd()
for _ in range(10):
    if (candidate / 'pyproject.toml').exists():
        break
    candidate = candidate.parent

os.chdir(candidate)
PROJECT_ROOT = Path.cwd()
print('Working directory:', PROJECT_ROOT)

H5_PATH = PROJECT_ROOT / 'data' / 'raw' / 'simple_mini_100s.h5'
print('Dataset :', H5_PATH)
print('Exists  :', H5_PATH.exists())

if not H5_PATH.exists():
    raise FileNotFoundError(
        f'Dataset not found: {H5_PATH}\n'
        'Generate it first with:\n'
        '  python scripts/run_simulation.py --config configs/simulations/simple_mini.yaml --n_samples 100'
    )

## Cell 2 — Imports

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib.pyplot as plt
from spectangle.simulations.io import load_simulation
from spectangle.models.unet import UNet2Dto3D

print('Imports OK')

## Cell 3 — Dataset class and instantiation

In [ ]:
class H5MiniDataset(Dataset):
    def __init__(self, path):
        data = load_simulation(str(path))
        self.samples = data['samples']
        meta = data['metadata']
        self.pad_y = int(meta['pad_y'])
        self.pad_x = int(meta['pad_x'])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        spec = s['spectrograms'].astype(np.float32)
        cube = s['cube'].astype(np.float32)
        direct = s.get('direct_image')
        if direct is not None:
            x = np.concatenate([spec, direct[None, ...]], axis=0)
        else:
            x = spec
        return torch.from_numpy(x), torch.from_numpy(cube)


# Use the absolute path resolved in Cell 1
ds = H5MiniDataset(H5_PATH)
print(f'Dataset loaded: {len(ds)} samples  pad_x={ds.pad_x}  pad_y={ds.pad_y}')

## Cell 4 — DataLoader + model

In [ ]:
loader = DataLoader(ds, batch_size=4, shuffle=True)

model = UNet2Dto3D(in_channels=4, n_lambda=128, base_features=16, depth=3, scene_shape=(128, 128))
opt = torch.optim.Adam(model.parameters(), lr=1e-4)

x_batch, y_batch = next(iter(loader))
print('x batch shape:', x_batch.shape)
print('y batch shape:', y_batch.shape)

## Cell 5 — Quick data visualisation

In [ ]:
spec0  = x_batch[0, 0].numpy()
broad0 = y_batch[0].numpy().sum(axis=0)

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(spec0,  origin='lower', cmap='inferno')
ax[0].set_title('Spectrogram ch0 (batch 0)')
ax[0].axis('off')
ax[1].imshow(broad0, origin='lower', cmap='gray')
ax[1].set_title('Ground-truth broadband (batch 0)')
ax[1].axis('off')
plt.tight_layout()
plt.show()

## Cell 6 — Training loop (3 epochs)

In [ ]:
loss_history = []
for epoch in range(3):
    for x, y in loader:
        pred = model(x)
        loss = torch.nn.functional.mse_loss(pred, y)
        opt.zero_grad()
        loss.backward()
        opt.step()
    loss_history.append(loss.item())
    print(f'Epoch {epoch}  loss={loss.item():.6f}')

plt.figure(figsize=(5, 3))
plt.plot(loss_history, marker='o')
plt.xlabel('Epoch')
plt.ylabel('MSE loss')
plt.title('Training loss (toy run)')
plt.grid(True)
plt.tight_layout()
plt.show()

## Cell 7 — Reconstruction visualisation

In [ ]:
model.eval()
with torch.no_grad():
    x0, y0 = ds[0]
    pred0 = model(x0.unsqueeze(0)).squeeze(0).cpu().numpy()  # (n_lambda, ny, nx)

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(y0.numpy().sum(axis=0),    origin='lower', cmap='gray')
ax[0].set_title('GT broadband');  ax[0].axis('off')
ax[1].imshow(pred0.sum(axis=0),         origin='lower', cmap='gray')
ax[1].set_title('Pred broadband'); ax[1].axis('off')
ax[2].imshow(np.abs(pred0.sum(axis=0) - y0.numpy().sum(axis=0)), origin='lower', cmap='magma')
ax[2].set_title('Absolute residual'); ax[2].axis('off')
plt.tight_layout()
plt.show()

If you'd like, I can convert the toy training loop to a fully featured
training script (with checkpointing, LR scheduler and mixed-precision), and
add a callback to visualise reconstructions at the end of each epoch.